# XGBWW Random-10 Long-Run Alpha Tracking (debug-friendly rewrite)

This notebook is split into small cells so you can debug each phase independently:
1. choose/verify 10 datasets and persist that list;
2. train one model per dataset with exhaustive logging per round;
3. compute WeightWatcher metrics (W1/W2/W7/W8/W9/W10) and train/test accuracy;
4. plot train/test accuracy and alphas vs round.


In [ ]:
# 1) Runtime setup and persistent output folders
from pathlib import Path
import os
import sys

RUNTIME_ROOT = Path.cwd()
OUTPUT_ROOT = Path('./random10_longrun_alpha_tracking')
REGISTRY_DIR = OUTPUT_ROOT / 'registry'
RUN_DIR = OUTPUT_ROOT / 'run'
PLOTS_DIR = OUTPUT_ROOT / 'plots'

for p in [OUTPUT_ROOT, REGISTRY_DIR, RUN_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Working directory:', RUNTIME_ROOT)
print('Output root:', OUTPUT_ROOT.resolve())


In [ ]:
# Colab/bootstrap installs
INCLUDE_KEEL_DATASETS = False  # Optional dataset source; default off for smoother Colab installs

%pip install -q xgboost weightwatcher torch scikit-learn pandas matplotlib seaborn scipy feather-format pyarrow openml pmlb requests
if INCLUDE_KEEL_DATASETS:
    %pip install -q keel-ds
else:
    print('[INFO] INCLUDE_KEEL_DATASETS=False -> KEEL datasets will be excluded from the registry scan.')

import pathlib
import shutil
import subprocess
import sys
import importlib


def clone_or_update(repo_url: str, target_dir: str, branch: str = 'main') -> None:
    target = pathlib.Path(target_dir)
    if (target / '.git').exists():
        subprocess.run(['git', '-C', str(target), 'fetch', '--depth', '1', 'origin', branch], check=True)
        subprocess.run(['git', '-C', str(target), 'reset', '--hard', 'FETCH_HEAD'], check=True)
    else:
        if target.exists():
            shutil.rmtree(target)
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(target)], check=True)

clone_or_update('https://github.com/CalculatedContent/xgboost2ww.git', '/content/repo_xgboost2ww')
clone_or_update('https://github.com/CalculatedContent/xgbwwdata.git', '/content/repo_xgbwwdata')

# Editable source installs from the cloned repos
%pip install -q -e /content/repo_xgboost2ww --no-build-isolation --no-deps
%pip install -q -e /content/repo_xgbwwdata --no-build-isolation --no-deps

importlib.invalidate_caches()
for mod_name in list(sys.modules):
    if mod_name in {'xgboost2ww', 'xgbwwdata'} or mod_name.startswith('xgboost2ww.') or mod_name.startswith('xgbwwdata.'):
        del sys.modules[mod_name]

xgboost = importlib.import_module('xgboost')
weightwatcher = importlib.import_module('weightwatcher')
torch = importlib.import_module('torch')
xgboost2ww = importlib.import_module('xgboost2ww')
xgbwwdata = importlib.import_module('xgbwwdata')

print('xgboost:', getattr(xgboost, '__version__', 'unknown'))
print('weightwatcher:', getattr(weightwatcher, '__version__', 'unknown'))
print('torch:', getattr(torch, '__version__', 'unknown'))
print('xgboost2ww location:', pathlib.Path(xgboost2ww.__file__).resolve())
print('xgbwwdata location:', pathlib.Path(xgbwwdata.__file__).resolve())
print('xgboost2ww installed from source')
print('xgbwwdata installed from source')


In [ ]:
# 3) Imports
import json
import time
import traceback
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import weightwatcher as ww

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from xgboost2ww import convert
from xgbwwdata import Filters, load_dataset

try:
    from xgbwwdata import scan_datasets as xgbww_scan_datasets
except Exception:
    xgbww_scan_datasets = None

try:
    from xgbwwdata import list_datasets as xgbww_list_datasets
except Exception:
    xgbww_list_datasets = None

print('Imports loaded.')


In [ ]:
# 4) Global config
RANDOM_STATE = 42
DATASET_SAMPLE_SIZE = 10
TOTAL_ROUNDS = 1000
CHUNK_SIZE = 25
N_STEPS = TOTAL_ROUNDS // CHUNK_SIZE
TEST_SIZE = 0.2

W_LIST = ['W1', 'W2', 'W7', 'W8', 'W9', 'W10']
SELECTED_DATASET_UIDS = []  # optionally pin exact dataset_uids
REUSE_EXISTING_SAMPLE = True
INCLUDE_KEEL_DATASETS = False
MAX_DENSE_ELEMENTS = int(2e8)
RESUME_FROM_CHECKPOINT = True
FORCE_RESTART_ALL = False

# lowercase aliases for compatibility with older cells/snippets
registry_dir = REGISTRY_DIR

SAMPLED_REGISTRY_CSV = REGISTRY_DIR / 'random10_registry.csv'
SAMPLED_REGISTRY_FEATHER = REGISTRY_DIR / 'random10_registry.feather'
TRAINING_LOG_CSV = RUN_DIR / 'training_metrics_longrun.csv'
TRAINING_LOG_FEATHER = RUN_DIR / 'training_metrics_longrun.feather'
ERROR_LOG_CSV = RUN_DIR / 'training_errors.csv'

np.random.seed(RANDOM_STATE)
print(f'N_STEPS={N_STEPS}, TOTAL_ROUNDS={TOTAL_ROUNDS}, CHUNK_SIZE={CHUNK_SIZE}')


In [ ]:
# xgbwwdata scan / registry build

import inspect

def _materialize_registry(reg):
    if isinstance(reg, pd.DataFrame):
        return reg
    if hasattr(reg, 'to_pandas'):
        return reg.to_pandas()
    return pd.DataFrame(list(reg))

def _call_xgbww_scan(api_func, filters: Filters, sources):
    sig = inspect.signature(api_func)
    kwargs = {}
    if 'filters' in sig.parameters:
        kwargs['filters'] = filters
    if 'sources' in sig.parameters:
        kwargs['sources'] = sources
    if 'smoke_train' in sig.parameters:
        kwargs['smoke_train'] = True
    if 'random_state' in sig.parameters:
        kwargs['random_state'] = RANDOM_STATE
    if 'log_every' in sig.parameters:
        kwargs['log_every'] = 25
    return api_func(**kwargs) if kwargs else api_func()

def build_registry_df(filters: Filters, sources) -> pd.DataFrame:
    if xgbww_scan_datasets is not None:
        return _materialize_registry(_call_xgbww_scan(xgbww_scan_datasets, filters, sources))
    if xgbww_list_datasets is not None:
        return _materialize_registry(_call_xgbww_scan(xgbww_list_datasets, filters, sources))
    try:
        from xgbwwdata import get_registry
        return _materialize_registry(_call_xgbww_scan(get_registry, filters, sources))
    except Exception as e:
        raise RuntimeError('No compatible xgbwwdata scan/list API found in this environment.') from e

filters = Filters(
    min_rows=200,
    max_rows=60000,
    max_features=50000,
    max_dense_elements=MAX_DENSE_ELEMENTS,
    preprocess=True,
)

SCAN_SOURCES = ['openml', 'pmlb', 'amlb', 'libsvm']
if INCLUDE_KEEL_DATASETS:
    SCAN_SOURCES.append('keel')

print('Scanning sources:', SCAN_SOURCES)

full_registry_csv = registry_dir / 'full_registry.csv'
full_registry_feather = registry_dir / 'full_registry.feather'

if RESUME_FROM_CHECKPOINT and full_registry_csv.exists() and not FORCE_RESTART_ALL:
    full_registry_df = pd.read_csv(full_registry_csv)
    print(f'Loaded cached full registry: {len(full_registry_df)} rows from {full_registry_csv}')
else:
    full_registry_df = build_registry_df(filters, SCAN_SOURCES)
    full_registry_df.to_csv(full_registry_csv, index=False)
    try:
        full_registry_df.reset_index(drop=True).to_feather(full_registry_feather)
    except Exception as e:
        print('[WARN] Feather save failed for full registry:', e)
    print(f'Scanned full registry: {len(full_registry_df)} rows')

if full_registry_df.empty:
    raise RuntimeError(f'Registry scan returned 0 rows for sources={SCAN_SOURCES}. No models can be trained.')

registry_df = full_registry_df.copy()
if 'task_type' in registry_df.columns:
    registry_df = registry_df[registry_df['task_type'].astype(str).str.contains('class', case=False, na=False)]
elif 'task' in registry_df.columns:
    registry_df = registry_df[registry_df['task'].astype(str).str.contains('class', case=False, na=False)]

if 'n_classes' in registry_df.columns:
    registry_df = registry_df[
        (registry_df['n_classes'].fillna(0) >= 2) &
        (registry_df['n_classes'].fillna(0) <= 20)
    ]

registry_df = registry_df.reset_index(drop=True)
print('Filtered registry rows:', len(registry_df))

# Random sample of 10 datasets (persist + reuse)
if REUSE_EXISTING_SAMPLE and SAMPLED_REGISTRY_CSV.exists() and not SELECTED_DATASET_UIDS:
    sampled_registry = pd.read_csv(SAMPLED_REGISTRY_CSV)
    print(f'Reused sampled registry from disk: {SAMPLED_REGISTRY_CSV}')
else:
    if SELECTED_DATASET_UIDS:
        sampled_registry = registry_df[registry_df['dataset_uid'].isin(SELECTED_DATASET_UIDS)].copy()
    else:
        sampled_registry = registry_df.sample(n=min(DATASET_SAMPLE_SIZE, len(registry_df)), random_state=RANDOM_STATE).copy()
    sampled_registry = sampled_registry.reset_index(drop=True)
    sampled_registry.to_csv(SAMPLED_REGISTRY_CSV, index=False)
    try:
        sampled_registry.to_feather(SAMPLED_REGISTRY_FEATHER)
    except Exception as e:
        print('[WARN] Could not save feather registry:', e)

print('Selected dataset count:', len(sampled_registry))
print('Selected columns:', list(sampled_registry.columns))
display(sampled_registry)


In [ ]:
# 6) Helpers for preprocessing, WW extraction, and structured logging

def detect_task_type(y):
    y_np = np.asarray(y)
    unique = np.unique(y_np)
    n_classes = len(unique)
    if n_classes < 2:
        return 'invalid', n_classes, y_np
    task = 'binary' if n_classes == 2 else 'multiclass'
    return task, n_classes, y_np


def preprocess_split(X, y):
    strat = y if len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=strat
    )
    scaler = StandardScaler(with_mean=False) if sparse.issparse(X_train) else StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)
    return X_train_s, X_test_s, np.asarray(y_train), np.asarray(y_test)


def build_params(task, n_classes):
    params = {
        'booster': 'gbtree',
        'eta': 0.05,
        'max_depth': 6,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'tree_method': 'hist',
        'seed': RANDOM_STATE,
    }
    if task == 'binary':
        params.update({'objective': 'binary:logistic', 'eval_metric': 'logloss'})
    else:
        params.update({'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'num_class': int(n_classes)})
    return params


def prediction_to_accuracy(bst, dmat, y_true, task):
    y_prob = bst.predict(dmat)
    if task == 'binary':
        y_pred = (y_prob >= 0.5).astype(np.int32)
    else:
        y_pred = np.argmax(y_prob, axis=1).astype(np.int32)
    return float(accuracy_score(y_true, y_pred))


def build_layer_for_W(bst, W_name, current_round, X_train_for_convert, y_train_np, params):
    multiclass_mode = 'avg' if int(params.get('num_class', 2)) > 2 else 'error'
    return convert(
        model=bst,
        data=X_train_for_convert,
        labels=y_train_np,
        W=W_name,
        return_type='torch',
        nfolds=5,
        t_points=min(current_round, 160),
        random_state=RANDOM_STATE,
        train_params=params,
        num_boost_round=current_round,
        multiclass=multiclass_mode,
    )


def ww_alphas_for_all_Ws(model, X_train, y_train, params, boost_round):
    out = {}
    for W in W_LIST:
        key = f'alpha_{W}'
        try:
            converted = build_layer_for_W(model, W, boost_round, X_train, y_train, params)
            details = ww.WeightWatcher(model=converted).analyze(randomize=True, detX=True)
            alpha_val = float(pd.to_numeric(details.get('alpha', pd.Series(dtype=float)), errors='coerce').dropna().mean())
            out[key] = alpha_val
        except Exception:
            out[key] = np.nan
    return out


def append_error(err_rows, dataset_uid, stage, exc):
    err_rows.append({
        'dataset_uid': str(dataset_uid),
        'stage': stage,
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback': traceback.format_exc(),
        'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    })


In [ ]:
# 7) Train each model one-by-one with exhaustive per-round logging
all_rows = []
error_rows = []

if sampled_registry.empty:
    raise RuntimeError('sampled_registry is empty. No datasets were selected, so the training loop would be a no-op.')

for i, row in sampled_registry.iterrows():
    dataset_uid = row.get('dataset_uid', f'row_{i}')
    source = row.get('source', 'unknown')
    print(f'\n===== [{i+1}/{len(sampled_registry)}] dataset_uid={dataset_uid} source={source} =====')

    try:
        X, y, meta = load_dataset(dataset_uid, filters=filters)
        task, n_classes, y_np = detect_task_type(y)
        if task == 'invalid':
            raise ValueError(f'Invalid target for dataset_uid={dataset_uid}')

        X_train, X_test, y_train, y_test = preprocess_split(X, y_np)
        dtrain, dtest = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_test, label=y_test)
        params = build_params(task, n_classes)

        bst = None
        for step in range(1, N_STEPS + 1):
            round_end = step * CHUNK_SIZE
            t0 = time.time()

            bst = xgb.train(params=params, dtrain=dtrain, num_boost_round=CHUNK_SIZE, xgb_model=bst, verbose_eval=False)
            train_acc = prediction_to_accuracy(bst, dtrain, y_train, task)
            test_acc = prediction_to_accuracy(bst, dtest, y_test, task)
            ww_metrics = ww_alphas_for_all_Ws(bst, X_train, y_train, params, round_end)

            row_out = {
                'dataset_uid': str(dataset_uid),
                'source': str(source),
                'step': int(step),
                'round': int(round_end),
                'train_accuracy': float(train_acc),
                'test_accuracy': float(test_acc),
                'elapsed_sec': float(time.time() - t0),
            }
            row_out.update(ww_metrics)
            all_rows.append(row_out)

            print(
                f"step={step:03d}/{N_STEPS} round={round_end:4d} "
                f"train_acc={train_acc:.4f} test_acc={test_acc:.4f} "
                f"alpha_W7={row_out.get('alpha_W7', np.nan):.4f} "
                f"elapsed={row_out['elapsed_sec']:.2f}s"
            )

        print(f'[DONE] dataset_uid={dataset_uid}')

    except Exception as e:
        append_error(error_rows, dataset_uid, 'train_loop', e)
        print(f'[ERROR] dataset_uid={dataset_uid}: {type(e).__name__}: {e}')

training_df = pd.DataFrame(all_rows)
training_df.to_csv(TRAINING_LOG_CSV, index=False)
try:
    training_df.to_feather(TRAINING_LOG_FEATHER)
except Exception as e:
    print('[WARN] Could not save training log feather:', e)

errors_df = pd.DataFrame(error_rows)
if not errors_df.empty:
    errors_df.to_csv(ERROR_LOG_CSV, index=False)

print('Saved training log:', TRAINING_LOG_CSV)
print('Training rows:', len(training_df), '| Error rows:', len(errors_df))
display(training_df.head(20))
display(errors_df.head(20) if not errors_df.empty else pd.DataFrame(columns=['dataset_uid', 'stage', 'error_type', 'error_message']))


In [ ]:
# 8) Plot per-dataset metrics after training (debug in a separate cell)
if 'training_df' not in globals() or training_df.empty:
    if TRAINING_LOG_CSV.exists():
        training_df = pd.read_csv(TRAINING_LOG_CSV)
        print('Loaded training_df from disk:', TRAINING_LOG_CSV)
    else:
        raise RuntimeError('No training dataframe available. Run the training cell first.')

for dataset_uid, g in training_df.groupby('dataset_uid'):
    g = g.sort_values('round').reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(g['round'], g['train_accuracy'], marker='o', label='train_accuracy')
    axes[0].plot(g['round'], g['test_accuracy'], marker='o', label='test_accuracy')
    axes[0].set_title(f'Accuracy vs Round | {dataset_uid}')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for W in W_LIST:
        col = f'alpha_{W}'
        if col in g.columns:
            axes[1].plot(g['round'], g[col], marker='.', label=col)
    axes[1].set_title(f'WW Alpha vs Round | {dataset_uid}')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Alpha')
    axes[1].legend(ncol=2, fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    png_path = PLOTS_DIR / f'{dataset_uid}_accuracy_alpha_vs_round.png'
    plt.savefig(png_path, dpi=140)
    plt.show()
    plt.close(fig)

    print('Saved plot:', png_path)


In [ ]:
# 9) Optional quick aggregate checks
if 'training_df' not in globals() or training_df.empty:
    training_df = pd.read_csv(TRAINING_LOG_CSV)

best_rows = training_df.loc[training_df.groupby('dataset_uid')['test_accuracy'].idxmax()].sort_values('test_accuracy', ascending=False)
display(best_rows[['dataset_uid', 'round', 'train_accuracy', 'test_accuracy'] + [f'alpha_{w}' for w in W_LIST if f'alpha_{w}' in best_rows.columns]])

summary = {
    'n_datasets_requested': DATASET_SAMPLE_SIZE,
    'n_datasets_trained': int(training_df['dataset_uid'].nunique()) if not training_df.empty else 0,
    'n_rows': int(len(training_df)),
    'training_csv': str(TRAINING_LOG_CSV),
    'registry_csv': str(SAMPLED_REGISTRY_CSV),
    'errors_csv': str(ERROR_LOG_CSV) if ERROR_LOG_CSV.exists() else None,
}
print(json.dumps(summary, indent=2))
